# Get to Know a Dataset: ALADIN Czech Republic 1 km Weather Forecasts

This notebook is a guided tour of the [ALADIN Czech Republic 1 km Weather Forecasts](https://registry.opendata.aws/aladin-cz-forecast) dataset: hourly 72-hour forecasts from the ALADIN numerical weather prediction model of the Czech Hydrometeorological Institute (ČHMÚ), archived as cloud-optimized Zarr. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

Data source: Czech Hydrometeorological Institute (ČHMÚ), [opendata.chmi.cz](https://opendata.chmi.cz/), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Full documentation: [README](README.md).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

All data sit under the `meteo_data/` prefix. Below that there is one prefix per calendar month (`YYYYMM`). Each month contains **one Zarr store per parameter**, up to 24 stores, for example `meteo_data/202602/CLSTEMPERATURE.zarr`.

```
s3://DOPLNIT-bucket-name/
  README.txt
  meteo_data/
    202506/
      CLPVEIND_MOD_XFU.zarr/
        zarr.json
        CLPVEIND_MOD_XFU/   heightAboveGround/   latitude/   longitude/
        step/   time/   valid_time/
      CLSHUMI_RELATIVE.zarr/
      CLSTEMPERATURE.zarr/
      ...
    202507/
    ...
```

A Zarr store is a "directory" of many small objects: a `zarr.json` metadata document, plus one object per chunk of every array. You normally never list or read these objects yourself. A Zarr library opens the store and fetches only the chunks it needs.

Before starting, here are the questions this notebook answers:
1. How are prefixes and objects organized? (this section)
2. What is the data format and how do I work with it?
3. How do I load a piece of the data?
4. What does the data look like?
5. One question we answered with the data
6. An open challenge for the community

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# xarray >= 2025.1
# zarr >= 3.0
# s3fs >= 2025.1
# dask >= 2025.1
# pandas >= 2.2
# matplotlib >= 3.9

import numpy as np
import pandas as pd
import xarray as xr
import s3fs
import matplotlib.pyplot as plt

First we set the location of the dataset and create an **anonymous** S3 filesystem. The bucket is public, so you do not need an AWS account or credentials. We then list the top-level prefixes.

In [ ]:
BUCKET = "DOPLNIT-bucket-name"           # DOPLNIT: final bucket name
ROOT = f"{BUCKET}/meteo_data"

fs = s3fs.S3FileSystem(anon=True)        # unsigned requests, no credentials needed

for item in fs.ls(BUCKET):
    print(item)

Inside `meteo_data/` there is one prefix per month (`YYYYMM`):

In [ ]:
for item in fs.ls(ROOT):
    print(item)

Each month holds one Zarr store per parameter. A store contains the `zarr.json` metadata, the coordinate arrays (`time`, `step`, `latitude`, `longitude`, `valid_time`, ...) and the data array named after the parameter:

In [ ]:
MONTH = "202602"                                  # month prefix, YYYYMM
MONTH_ISO = f"{MONTH[:4]}-{MONTH[4:]}"            # "2026-02", handy for date selections

for item in fs.ls(f"{ROOT}/{MONTH}"):
    print(item)

print()
for item in fs.ls(f"{ROOT}/{MONTH}/CLSTEMPERATURE.zarr"):
    print(item)

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

All data are stored as **[Zarr v3](https://zarr.dev/)**. Zarr is a format for chunked, compressed N-dimensional arrays. Each chunk is its own object in S3, and a small JSON document describes the arrays.

ČHMÚ publishes the model output as GRIB files, one file per parameter and run, available for only a few days. We converted them to Zarr because:
- **you only download what you need**. A time series at one point, or a map at one hour, reads a few chunks instead of whole files.
- **many readers can work in parallel**. Each chunk is an independent S3 object, which suits Dask, AWS Lambda or many EC2 workers.
- **the metadata are consolidated**. Opening a store is a single request, and coordinates, units and chunking are known before any data are read.
- it is supported natively by `xarray`, which is the standard tool in the weather and climate Python ecosystem.

**Layout of every store:**

| | |
|---|---|
| data variable | exactly 1, named after the parameter, `float32` |
| dimensions | `(time, step, latitude, longitude)` = `(model runs in the month, 72, 290, 501)` |
| chunks | `time=1, step=12, latitude=97, longitude=126` |
| grid | regular lat/lon, 48.5–51.1 °N × 12.0–19.0 °E, ~1 km |
| compression | Zstd |

**Things specific to this dataset that you should know:**
- `time` is the **model run (initialization) time** (00/06/12/18 UTC). `step` is the **lead time in hours**, stored as a float, *not* a timedelta. `valid_time = time + step` is provided as a 2-D coordinate.
- Instantaneous parameters (temperature, wind, ...) have `step = 0..71`. Accumulated parameters (precipitation, radiation, sunshine) have `step = 1..72` and are **summed from the start of the run**. Take the difference along `step` to get hourly values.
- The `units` / `long_name` attributes inherited from GRIB are often wrong (ČHMÚ uses local GRIB tables). Use the parameter table in the [documentation](README.md).
- Coverage is tracked per parameter. A day that is missing for one parameter is absent from that store's `time` axis (not filled with NaN), while other parameters can have it. Stores of the same month can therefore have different `time` axes.

**Tools:** `xarray` + `zarr` + `s3fs` (+ `dask` for lazy, parallel reads) in Python. The [zarr](https://zarr.dev/) ecosystem also covers Julia, R, JavaScript and Rust. For larger analyses, run your code in the **same AWS region** as the bucket, on Amazon EC2 or Amazon SageMaker AI. This gives the fastest reads.

### Q: Can you show us an example of downloading and loading data from your dataset?

We open the 2 m temperature store for February 2026. `xr.open_zarr` reads only the metadata, and no data are downloaded yet.

In [ ]:
def open_param(month, param):
    """Open one parameter for one month as a lazy xarray Dataset."""
    return xr.open_zarr(f"s3://{ROOT}/{month}/{param}.zarr",
                        consolidated=True, storage_options={"anon": True})

ds = open_param(MONTH, "CLSTEMPERATURE")
ds

Each run time in `time` carries a full 72-hour forecast. The forecast for run `time[i]`, lead `step[j]` is valid at `valid_time[i, j]`:

In [ ]:
print("first runs:", ds.time.values[:4])
print("steps (hours):", ds.step.values[:5], "...", ds.step.values[-1])
print("valid times of the first run:", ds.valid_time.isel(time=0).values[[0, 1, -1]])

# NOTE: step is a float in hours -> convert with unit="h"
print(pd.to_timedelta(ds.step.values[:3], unit="h"))

Now we actually download some data. We take the 2 m temperature forecast for Prague from one run. That is 72 values, read from 6 chunks.

In [ ]:
PRAGUE = dict(latitude=50.08, longitude=14.42)

run = ds.time.values[0]
t2m_prague = (ds.CLSTEMPERATURE
              .sel(time=run)
              .sel(PRAGUE, method="nearest")
              .load() - 273.15)          # K -> °C

t2m_prague.to_dataframe(name="t2m_degC")[["valid_time", "t2m_degC"]].head(8)

The `time` axis lists only the runs that are present in this store. Coverage differs between parameters, so check it for each store you use:

In [ ]:
expected = pd.date_range(f"{MONTH_ISO}-01", periods=ds.indexes["time"][0].days_in_month * 4, freq="6h")
missing = expected.difference(ds.indexes["time"])
print(f"{len(ds.time)} of {len(expected)} runs present; missing: {len(missing)}")
print(missing[:10])

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

**1. Maps.** We pick one run and lead time and show 2 m temperature, hourly precipitation and total cloud cover over the whole Czech Republic at 1 km resolution. One field is 12 chunks (~7 MB uncompressed).

Precipitation is accumulated from the start of the run, so the precipitation for the hour ending at `step = h` is `acc(h) - acc(h-1)`.

In [ ]:
# Parameters used in the maps and the meteogram below. Coverage differs per parameter,
# so we pick a run that exists in all of them (only metadata are read here).
PARAMS = ["CLSTEMPERATURE", "SURFPREC_TOTAL", "SURFNEBUL_TOTALE",
          "CLSHUMI_RELATIVE", "SURFSNOWFALL", "CLSWIND_SPEED"]
common_runs = sorted(set.intersection(*(set(open_param(MONTH, p).time.values) for p in PARAMS)))
print(f"{len(common_runs)} runs are present in all {len(PARAMS)} parameters")

RUN = common_runs[len(common_runs) // 2]    # a common run in the middle of the month
LEAD = 24                                   # hours

t2m  = open_param(MONTH, "CLSTEMPERATURE").CLSTEMPERATURE.sel(time=RUN, step=LEAD) - 273.15
prec = open_param(MONTH, "SURFPREC_TOTAL").SURFPREC_TOTAL.sel(time=RUN)
prec_1h = (prec.sel(step=LEAD) - prec.sel(step=LEAD - 1)).clip(min=0)
cloud = open_param(MONTH, "SURFNEBUL_TOTALE").SURFNEBUL_TOTALE.sel(time=RUN, step=LEAD)

fields = [
    (t2m,     "2 m temperature [°C]",       "RdYlBu_r", None, None),
    (prec_1h.where(prec_1h > 0.05), "1-h precipitation [mm]", "viridis_r", 0, 5),
    (cloud,   "total cloud cover [0–1]",     "Blues_r",  0, 1),
]
valid = pd.Timestamp(RUN) + pd.Timedelta(hours=LEAD)

fig, axes = plt.subplots(3, 1, figsize=(9, 13), constrained_layout=True)
for ax, (da, title, cmap, vmin, vmax) in zip(axes, fields):
    im = ax.pcolormesh(da.longitude, da.latitude, da, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
    ax.plot(PRAGUE["longitude"], PRAGUE["latitude"], "k*", ms=10)
    ax.set_title(f"{title}  —  valid {valid:%Y-%m-%d %H:%M} UTC (run {pd.Timestamp(RUN):%d %b %H} UTC +{LEAD} h)")
    ax.set_aspect(1 / np.cos(np.deg2rad(49.8)))   # approx. equal-distance aspect at this latitude
    ax.set_xlabel("longitude [°E]"); ax.set_ylabel("latitude [°N]")
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

**2. Meteogram.** We show the full 72-hour forecast for Prague from the same run. Different parameters have different `step` axes (0–71 or 1–72), so we line them up on `valid_time`, not on the step index.

In [ ]:
def point_series(param, run=RUN, point=PRAGUE):
    da = open_param(MONTH, param)[param].sel(time=run).sel(point, method="nearest").load()
    return pd.Series(da.values, index=pd.DatetimeIndex(da.valid_time.values), name=param)

t   = point_series("CLSTEMPERATURE") - 273.15
rh  = point_series("CLSHUMI_RELATIVE") * 100            # stored as fraction 0-1
pr  = point_series("SURFPREC_TOTAL").diff().fillna(point_series("SURFPREC_TOTAL").iloc[0]).clip(lower=0)
sn  = point_series("SURFSNOWFALL").diff().fillna(point_series("SURFSNOWFALL").iloc[0]).clip(lower=0)
cc  = point_series("SURFNEBUL_TOTALE") * 100
ws  = point_series("CLSWIND_SPEED")

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
a1.plot(t.index, t, color="tab:red"); a1.set_ylabel("T 2 m [°C]")
a1b = a1.twinx(); a1b.plot(rh.index, rh, color="tab:blue", alpha=0.5); a1b.set_ylabel("RH [%]", color="tab:blue")
a2.bar(pr.index, pr, width=1/30, color="tab:green", label="total")
a2.bar(sn.index, sn, width=1/30, color="tab:cyan", label="of which snow")
a2.set_ylabel("precip. [mm/h]"); a2.legend(loc="upper left")
a3.fill_between(cc.index, cc, color="grey", alpha=0.4); a3.set_ylabel("cloud cover [%]")
a3b = a3.twinx(); a3b.plot(ws.index, ws, color="k"); a3b.set_ylabel("wind 10 m [m/s]")
fig.suptitle(f"ALADIN CZ_1km meteogram for Prague — run {pd.Timestamp(RUN):%Y-%m-%d %H} UTC")
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**How quickly does the forecast for a given hour change as the lead time grows?**

For each hour, the archive holds up to 12 forecasts made at different times: the newest run 0 h ahead, then 6 h, 12 h, … up to 66 h ahead. We plot these forecasts for Prague over about a week. As a simple reference "truth" we use the model's own **analysis**, which is the `step = 0` value of the run valid at that hour. Against it, we measure how the error grows with lead time.

This is only a consistency check of the model against itself. Real verification needs observations (see the challenge below). It still shows how to work with the `time × step × valid_time` structure.

In [ ]:
# 2 m temperature at Prague for ~1 week of runs: 28 runs x 6 chunks are read
pt = (open_param(MONTH, "CLSTEMPERATURE").CLSTEMPERATURE
      .sel(PRAGUE, method="nearest")
      .sel(time=slice(f"{MONTH_ISO}-08", f"{MONTH_ISO}-14"))
      .load() - 273.15)

df = (pt.to_dataframe(name="t2m")
        .reset_index()[["time", "step", "valid_time", "t2m"]])
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
for run, g in df.groupby("time"):
    ax.plot(g.valid_time, g.t2m, lw=0.8, alpha=0.5)
analysis = df[df.step == 0].set_index("valid_time").t2m.sort_index()
ax.plot(analysis.index, analysis, "ko", ms=4, label="analysis (step 0 of each run)")
ax.set_ylabel("T 2 m [°C]"); ax.legend()
ax.set_title("Prague: every run's 72 h forecast (coloured) vs. analysis (black)")
plt.show()

In [ ]:
# Compare each forecast with the analysis valid at the same time
df["analysis"] = df.valid_time.map(analysis)
scored = df.dropna(subset=["analysis"]).query("step > 0")
mae = (scored.t2m - scored.analysis).abs().groupby(scored.step).mean()

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.plot(mae.index, mae.values, "o-")
ax.set_xlabel("lead time [h]"); ax.set_ylabel("mean |forecast − analysis| [K]")
ax.set_title("Prague 2 m temperature: forecast change vs. lead time")
ax.grid(alpha=0.3)
plt.show()
mae.round(2).to_frame("MAE_K").T

For this week in Prague, the difference to the analysis roughly doubles between the 6 h and the 60 h forecast (from about 0.8 K to 1.7 K). With one point and one week the curve is noisy. To get a robust answer, repeat this over many points and months. Each extra month is just another `open_param(...)` call, and the stores can be combined with `xr.concat(..., dim="time")`.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Challenge: How skilful are 1 km ALADIN forecasts of precipitation and thunderstorms, and can machine learning post-processing make them better?**

Kilometre-scale models resolve convective storms explicitly. That makes them valuable, and it also makes them hard to verify. A storm forecast 10 km away from where it happened is "wrong" at every grid point (the *double penalty*). This archive keeps every run and every forecast hour. You can use it to:

1. **Verify** hourly precipitation (`SURFPREC_TOTAL`, `SURFRAINFALL`, `SURFSNOWFALL`) and the lightning diagnostic (`SURFDIAG_FLASH`) against observations. Possible sources are ČHMÚ radar composites and station data, both published on [opendata.chmi.cz](https://opendata.chmi.cz/), or European lightning detection.
2. **Post-process.** Train a model (gradient boosting, U-Net, ...) that turns the raw forecast fields (CAPE, cloud cover, precipitation type, humidity, ...) into calibrated probabilities of precipitation or lightning at each grid point.

**Advice:**
- Use neighbourhood / spatial scores such as the Fractions Skill Score instead of point-to-point errors at 1 km.
- Accumulated fields are summed from the run start. Take the difference along `step`, and clip small negative values caused by GRIB packing.
- Align forecasts and observations on `valid_time`. When you combine several parameters, keep only the runs they all have (`xr.merge(..., join="inner")`).
- Do not trust the GRIB attributes. Use the parameter table in the [documentation](README.md).
- Run close to the data, in the bucket's AWS region. Use Dask to process many runs in parallel, one Zarr chunk per task.

We would love to hear about your results. Share them or ask questions at https://github.com/dandamasek/open-data-examples/issues.